# Agent2Agent (A2A) Protocol — A Working Notebook

This notebook builds a **real, runnable** A2A implementation from scratch —
no external agent servers, no network setup. Every request in here actually
executes, in-process, using `httpx`'s ASGI transport talking directly to a
`FastAPI` app in memory.

By the end you will have:

1. A hand-built **Agent Card**, **Task**, **Message/Part**, and **Artifact** model
2. A minimal but spec-shaped **A2A server** (`message/send`, `tasks/get`, `tasks/cancel`)
3. A **client** that discovers the agent, sends a task, and streams updates over SSE
4. A **multi-agent orchestration** demo — one orchestrator fanning a job out to two specialist agents in parallel
5. A basic **auth layer** and a **production checklist**

> Companion to the "Agent2Agent Protocol" animated HTML deck — this notebook is the
> "now go build it" half.


## 0. Setup

Everything here runs on top of four libraries:

| Library | Role |
|---|---|
| `fastapi` | implements the A2A server's HTTP surface |
| `httpx` | the client — talks to the FastAPI app **without opening a real socket**, via `ASGITransport` |
| `pydantic` | typed models for Agent Card / Task / Message / Part |
| `asyncio` | everything in A2A is async — tasks run, stream, and get canceled concurrently |

In a real deployment you'd run the server with `uvicorn app:app --host 0.0.0.0 --port 8080`
and the client would hit a real URL. Here we skip the network stack so the whole
notebook is self-contained and reproducible.


In [1]:
import subprocess, sys

def _pip_install(*pkgs):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *pkgs])
    except subprocess.CalledProcessError:
        # Some environments (e.g. system Python on Debian/Ubuntu) refuse installs
        # unless this flag is passed — harmless everywhere else.
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                                "--break-system-packages", *pkgs])

_pip_install("fastapi", "httpx", "pydantic", "uvicorn", "nest_asyncio")
print("Dependencies installed.")


error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

Dependencies installed.


In [2]:
import asyncio
import json
import time
import uuid
from enum import Enum
from typing import Any, Dict, List, Optional

import httpx
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse, StreamingResponse
from pydantic import BaseModel, Field

import nest_asyncio
nest_asyncio.apply()  # lets us use asyncio.run() safely inside Jupyter's own event loop

print("Environment ready.")


Environment ready.


## 1. The Vocabulary — Agent Card, Task, Message, Part, Artifact

Five typed objects are the entire vocabulary of A2A. We define them with
Pydantic so every request/response in this notebook is validated, not just
"probably right".


In [3]:
class TaskState(str, Enum):
    submitted       = "submitted"
    working         = "working"
    input_required  = "input-required"
    auth_required   = "auth-required"
    completed       = "completed"
    failed          = "failed"
    canceled        = "canceled"
    rejected        = "rejected"

TERMINAL_STATES = {TaskState.completed, TaskState.failed, TaskState.canceled, TaskState.rejected}


class Part(BaseModel):
    """A single typed chunk of content inside a Message or Artifact."""
    kind: str                     # "text" | "data" | "file"
    text: Optional[str] = None
    data: Optional[dict] = None
    file_uri: Optional[str] = None
    mime_type: Optional[str] = None


class Message(BaseModel):
    """One turn of dialogue — a list of Parts plus who sent it."""
    role: str                     # "user" | "agent"
    parts: List[Part]
    message_id: str = Field(default_factory=lambda: str(uuid.uuid4())[:8])


class Artifact(BaseModel):
    """A piece of output produced by a task."""
    name: str
    parts: List[Part]


class AgentSkill(BaseModel):
    id: str
    description: str
    input_modes: List[str] = ["text/plain"]
    output_modes: List[str] = ["application/json"]


class AgentCapabilities(BaseModel):
    streaming: bool = False
    push_notifications: bool = False


class AgentCard(BaseModel):
    """The 'business card' a client fetches before it ever sends a task."""
    name: str
    description: str
    url: str
    version: str
    capabilities: AgentCapabilities
    skills: List[AgentSkill]
    security_schemes: Dict[str, dict] = {}


class Task(BaseModel):
    id: str
    context_id: str
    status: TaskState
    history: List[Message] = []
    artifacts: List[Artifact] = []

print("Core A2A types defined:", ["TaskState", "Part", "Message", "Artifact", "AgentSkill", "AgentCard", "Task"])


Core A2A types defined: ['TaskState', 'Part', 'Message', 'Artifact', 'AgentSkill', 'AgentCard', 'Task']


In [4]:
# Quick sanity check: build one of each by hand and print it
example_card = AgentCard(
    name="refund-agent",
    description="Validates and issues refunds for e-commerce orders",
    url="https://agents.acme.co/refund",
    version="1.0.0",
    capabilities=AgentCapabilities(streaming=True, push_notifications=True),
    skills=[AgentSkill(id="process-refund", description="Issues a refund for an order id")],
)
print(example_card.model_dump_json(indent=2))


{
  "name": "refund-agent",
  "description": "Validates and issues refunds for e-commerce orders",
  "url": "https://agents.acme.co/refund",
  "version": "1.0.0",
  "capabilities": {
    "streaming": true,
    "push_notifications": true
  },
  "skills": [
    {
      "id": "process-refund",
      "description": "Issues a refund for an order id",
      "input_modes": [
        "text/plain"
      ],
      "output_modes": [
        "application/json"
      ]
    }
  ],
  "security_schemes": {}
}


## 2. A Minimal A2A Server

We'll build **`RefundAgentServer`**: a FastAPI app that implements exactly three
JSON-RPC methods, which cover the overwhelming majority of real A2A traffic:

| Method | Purpose |
|---|---|
| `message/send` | send a message, get back a task (blocking / synchronous style) |
| `message/stream` | same, but the response streams over Server-Sent Events |
| `tasks/get` | poll a task's current state |
| `tasks/cancel` | ask the agent to stop working on a task |

The Agent Card is also served as a plain file at the conventional
`/.well-known/agent-card.json` path — that's the entire discovery mechanism.


In [5]:
class InMemoryTaskStore:
    """Stand-in for a real database. Swap this for Redis/Postgres in production."""
    def __init__(self):
        self._tasks: Dict[str, Task] = {}

    def create(self, context_id: str, first_message: Message) -> Task:
        task = Task(
            id=str(uuid.uuid4())[:8],
            context_id=context_id,
            status=TaskState.submitted,
            history=[first_message],
        )
        self._tasks[task.id] = task
        return task

    def get(self, task_id: str) -> Optional[Task]:
        return self._tasks.get(task_id)

    def save(self, task: Task):
        self._tasks[task.id] = task


def build_agent_server(card: AgentCard, skill_handler):
    """
    Factory that wires up a FastAPI app implementing the A2A surface for a
    single skill handler. `skill_handler(order_text: str) -> dict` does the
    actual work and is the only thing that changes between agents.
    """
    app = FastAPI(title=card.name)
    store = InMemoryTaskStore()

    @app.get("/.well-known/agent-card.json")
    def get_card():
        return card.model_dump(by_alias=True)

    def _extract_text(message: dict) -> str:
        for p in message.get("parts", []):
            if p.get("kind") == "text":
                return p.get("text", "")
        return ""

    async def _run_skill(task: Task):
        """Simulates real work: moves the task through working -> completed,
        producing an artifact along the way."""
        task.status = TaskState.working
        store.save(task)
        await asyncio.sleep(0.05)  # pretend this takes a moment

        input_text = _extract_text(task.history[-1].model_dump())
        result = skill_handler(input_text)

        task.artifacts.append(Artifact(name="result", parts=[Part(kind="data", data=result)]))
        task.status = TaskState.completed
        store.save(task)
        return task

    @app.post("/")
    async def rpc(request: Request):
        body = await request.json()
        method, params, req_id = body.get("method"), body.get("params", {}), body.get("id")

        if method == "message/send":
            msg = Message(**params["message"])
            task = store.create(context_id=str(uuid.uuid4())[:8], first_message=msg)
            task = await _run_skill(task)
            return {"jsonrpc": "2.0", "id": req_id, "result": task.model_dump()}

        elif method == "tasks/get":
            task = store.get(params["id"])
            if task is None:
                return JSONResponse(
                    {"jsonrpc": "2.0", "id": req_id, "error": {"code": -32001, "message": "Task not found"}},
                    status_code=404,
                )
            return {"jsonrpc": "2.0", "id": req_id, "result": task.model_dump()}

        elif method == "tasks/cancel":
            task = store.get(params["id"])
            if task and task.status not in TERMINAL_STATES:
                task.status = TaskState.canceled
                store.save(task)
            return {"jsonrpc": "2.0", "id": req_id, "result": {"id": params["id"], "status": task.status if task else None}}

        return JSONResponse(
            {"jsonrpc": "2.0", "id": req_id, "error": {"code": -32601, "message": "Method not found"}},
            status_code=400,
        )

    @app.post("/stream")
    async def rpc_stream(request: Request):
        body = await request.json()
        msg = Message(**body["params"]["message"])
        task = store.create(context_id=str(uuid.uuid4())[:8], first_message=msg)

        async def event_gen():
            yield f"data: {json.dumps({'id': task.id, 'status': TaskState.working.value})}\n\n"
            task.status = TaskState.working
            store.save(task)
            await asyncio.sleep(0.03)

            input_text = _extract_text(msg.model_dump())
            result = skill_handler(input_text)
            yield f"data: {json.dumps({'id': task.id, 'artifact': result})}\n\n"
            await asyncio.sleep(0.03)

            task.artifacts.append(Artifact(name="result", parts=[Part(kind="data", data=result)]))
            task.status = TaskState.completed
            store.save(task)
            yield f"data: {json.dumps({'id': task.id, 'status': TaskState.completed.value})}\n\n"

        return StreamingResponse(event_gen(), media_type="text/event-stream")

    return app, store

print("build_agent_server() ready.")


build_agent_server() ready.


## 3. Standing Up the Refund Agent

`skill_handler` is deliberately the *only* agent-specific logic — everything
else above it (discovery, task state, streaming, RPC plumbing) is generic
A2A infrastructure you'd reuse for every agent you build.


In [6]:
def refund_skill(order_text: str) -> dict:
    # In a real agent this might call your payments provider, look up the
    # order, check refund eligibility, etc.
    order_id = "".join(ch for ch in order_text if ch.isdigit()) or "unknown"
    return {"order_id": order_id, "refund_status": "issued", "amount_usd": 42.00}


refund_card = AgentCard(
    name="refund-agent",
    description="Validates and issues refunds for e-commerce orders",
    url="https://agents.acme.co/refund",
    version="1.0.0",
    capabilities=AgentCapabilities(streaming=True, push_notifications=True),
    skills=[AgentSkill(id="process-refund", description="Issues a refund for an order id")],
)

refund_app, refund_store = build_agent_server(refund_card, refund_skill)
print("refund-agent is live (in-process).")


refund-agent is live (in-process).


## 4. The Client Side

The client never imports anything from `refund_app`'s implementation — only
its Agent Card. We use `httpx.ASGITransport` so this talks to the FastAPI app
directly in memory (swap this transport for a plain `base_url="https://..."`
client to hit a real deployed agent — nothing else in the client code changes).


In [7]:
async def get_client(app: FastAPI, base_url: str = "http://agent.local") -> httpx.AsyncClient:
    transport = httpx.ASGITransport(app=app)
    return httpx.AsyncClient(transport=transport, base_url=base_url)


async def discover(client: httpx.AsyncClient) -> AgentCard:
    r = await client.get("/.well-known/agent-card.json")
    r.raise_for_status()
    return AgentCard(**r.json())


async def send_message(client: httpx.AsyncClient, text: str) -> dict:
    payload = {
        "jsonrpc": "2.0",
        "id": str(uuid.uuid4())[:8],
        "method": "message/send",
        "params": {"message": {"role": "user", "parts": [{"kind": "text", "text": text}]}},
    }
    r = await client.post("/", json=payload)
    r.raise_for_status()
    return r.json()["result"]


async def demo_basic_call():
    client = await get_client(refund_app)
    card = await discover(client)
    print(f"Discovered: {card.name} v{card.version}  |  skills: {[s.id for s in card.skills]}\n")

    result = await send_message(client, "Please refund order #4471")
    print("Task ID   :", result["id"])
    print("Status    :", result["status"])
    print("Artifacts :", result["artifacts"])
    await client.aclose()

asyncio.run(demo_basic_call())


Discovered: refund-agent v1.0.0  |  skills: ['process-refund']

Task ID   : 141063ce
Status    : completed
Artifacts : [{'name': 'result', 'parts': [{'kind': 'data', 'text': None, 'data': {'order_id': '4471', 'refund_status': 'issued', 'amount_usd': 42.0}, 'file_uri': None, 'mime_type': None}]}]


## 5. Streaming with `message/stream`

Instead of waiting for one final response, the client opens an SSE
connection and watches the task's state change in real time — this is what
lets a UI show "thinking... → drafting... → done" instead of a blank spinner.


In [8]:
async def stream_message(client: httpx.AsyncClient, text: str):
    payload = {
        "jsonrpc": "2.0",
        "id": str(uuid.uuid4())[:8],
        "method": "message/stream",
        "params": {"message": {"role": "user", "parts": [{"kind": "text", "text": text}]}},
    }
    async with client.stream("POST", "/stream", json=payload) as resp:
        async for line in resp.aiter_lines():
            if line.startswith("data:"):
                event = json.loads(line[len("data:"):].strip())
                yield event


async def demo_streaming():
    client = await get_client(refund_app)
    async for event in stream_message(client, "Refund order #9981 please"):
        print(f"[t+{time.time():.2f}]  {event}")
    await client.aclose()

asyncio.run(demo_streaming())


[t+1785051219.92]  {'id': 'f165fc27', 'status': 'working'}
[t+1785051219.92]  {'id': 'f165fc27', 'artifact': {'order_id': '9981', 'refund_status': 'issued', 'amount_usd': 42.0}}
[t+1785051219.92]  {'id': 'f165fc27', 'status': 'completed'}


## 6. The Task Lifecycle, Made Visible

Let's watch a task walk its state machine explicitly, including a manual
`tasks/cancel` — this is the mechanism a long-running task relies on so it
doesn't run forever after a user changes their mind.


In [9]:
async def rpc_call(client: httpx.AsyncClient, method: str, params: dict) -> dict:
    payload = {"jsonrpc": "2.0", "id": str(uuid.uuid4())[:8], "method": method, "params": params}
    r = await client.post("/", json=payload)
    return r.json()


async def demo_cancel():
    client = await get_client(refund_app)

    sent = await rpc_call(client, "message/send", {
        "message": {"role": "user", "parts": [{"kind": "text", "text": "Refund order #1200"}]}
    })
    task_id = sent["result"]["id"]
    print("Created task:", task_id, "-> status:", sent["result"]["status"])

    # it already completed synchronously in our toy handler, so cancel is a no-op here —
    # but this is exactly the call a client makes on a task still in `working`
    canceled = await rpc_call(client, "tasks/cancel", {"id": task_id})
    print("Cancel response:", canceled["result"])

    fetched = await rpc_call(client, "tasks/get", {"id": task_id})
    print("Final state via tasks/get:", fetched["result"]["status"])
    await client.aclose()

asyncio.run(demo_cancel())


Created task: 23241018 -> status: completed
Cancel response: {'id': '23241018', 'status': 'completed'}
Final state via tasks/get: completed


## 7. Multi-Agent Orchestration

Now the interesting part: an **orchestrator** that fans one job out to two
independent specialist agents *in parallel*, then merges their artifacts —
the same pattern behind a "check price + check stock" checkout flow, or a
research agent that queries three data sources at once.


In [10]:
def pricing_skill(order_text: str) -> dict:
    return {"skill": "quote-price", "quote_usd": 129.99}

def inventory_skill(order_text: str) -> dict:
    return {"skill": "check-stock", "in_stock": True, "warehouse": "SEA-3"}

pricing_card = AgentCard(
    name="pricing-agent", description="Quotes prices", url="https://agents.acme.co/pricing",
    version="1.0.0", capabilities=AgentCapabilities(streaming=False),
    skills=[AgentSkill(id="quote-price", description="Quotes a price for an order")],
)
inventory_card = AgentCard(
    name="inventory-agent", description="Checks stock", url="https://agents.acme.co/inventory",
    version="1.0.0", capabilities=AgentCapabilities(streaming=False),
    skills=[AgentSkill(id="check-stock", description="Checks stock for an order")],
)

pricing_app, _ = build_agent_server(pricing_card, pricing_skill)
inventory_app, _ = build_agent_server(inventory_card, inventory_skill)


async def call_agent(app: FastAPI, text: str) -> dict:
    client = await get_client(app)
    try:
        result = await send_message(client, text)
        return result["artifacts"][0]["parts"][0]["data"]
    finally:
        await client.aclose()


async def orchestrate(order_text: str) -> dict:
    """The orchestrator's whole job: fan out, wait, merge."""
    pricing_result, inventory_result = await asyncio.gather(
        call_agent(pricing_app, order_text),
        call_agent(inventory_app, order_text),
    )
    return {"order": order_text, "pricing": pricing_result, "inventory": inventory_result}


final = asyncio.run(orchestrate("Order #5521 — wireless keyboard"))
print(json.dumps(final, indent=2))


{
  "order": "Order #5521 \u2014 wireless keyboard",
  "pricing": {
    "skill": "quote-price",
    "quote_usd": 129.99
  },
  "inventory": {
    "skill": "check-stock",
    "in_stock": true,
    "warehouse": "SEA-3"
  }
}


## 8. Adding Authentication

The Agent Card advertises which auth schemes it accepts via
`security_schemes`. Here we add a simple bearer-token check to show the
*mechanism* — swap the token check for real OAuth2/JWT validation in
production.


In [11]:
VALID_TOKENS = {"demo-token-123"}

def build_secure_agent_server(card: AgentCard, skill_handler):
    app, store = build_agent_server(card, skill_handler)

    @app.middleware("http")
    async def check_auth(request: Request, call_next):
        # Discovery stays public — you must be able to read the card before
        # you have a token for it.
        if request.url.path == "/.well-known/agent-card.json":
            return await call_next(request)

        token = request.headers.get("authorization", "").removeprefix("Bearer ").strip()
        if token not in VALID_TOKENS:
            return JSONResponse({"error": "unauthorized"}, status_code=401)
        return await call_next(request)

    return app, store


secure_card = refund_card.model_copy(update={
    "security_schemes": {"bearer": {"type": "http", "scheme": "bearer"}}
})
secure_app, _ = build_agent_server(secure_card, refund_skill)
# rebuild with the auth middleware wrapped around the same skill
secure_app, _ = build_secure_agent_server(secure_card, refund_skill)


async def demo_auth():
    client = await get_client(secure_app)

    # 1) no token -> rejected
    r = await client.post("/", json={"jsonrpc": "2.0", "id": "1", "method": "message/send",
                                      "params": {"message": {"role": "user", "parts": [{"kind": "text", "text": "test"}]}}})
    print("No token      ->", r.status_code, r.json())

    # 2) wrong token -> rejected
    r = await client.post("/", headers={"Authorization": "Bearer wrong-token"},
                           json={"jsonrpc": "2.0", "id": "2", "method": "message/send",
                                 "params": {"message": {"role": "user", "parts": [{"kind": "text", "text": "test"}]}}})
    print("Wrong token   ->", r.status_code, r.json())

    # 3) valid token -> works
    r = await client.post("/", headers={"Authorization": "Bearer demo-token-123"},
                           json={"jsonrpc": "2.0", "id": "3", "method": "message/send",
                                 "params": {"message": {"role": "user", "parts": [{"kind": "text", "text": "Refund order #77"}]}}})
    print("Valid token   ->", r.status_code, r.json()["result"]["status"])
    await client.aclose()

asyncio.run(demo_auth())


No token      -> 401 {'error': 'unauthorized'}
Wrong token   -> 401 {'error': 'unauthorized'}


Valid token   -> 200 completed


## 9. Idempotency: Why Retries Need Care

A2A clients will retry on timeouts. If `message/send` isn't idempotent, a
retried "issue refund" call can issue the refund **twice**. Below is the bug,
then the one-line-of-bookkeeping fix.


In [12]:
# --- the naive (buggy) version: every call does the work again ---
refund_ledger_naive = []

def naive_refund_handler(request_id: str, order_text: str):
    refund_ledger_naive.append(order_text)  # ALWAYS appends -> double refund on retry
    return {"status": "issued"}

naive_refund_handler("req-1", "order #900")
naive_refund_handler("req-1", "order #900")  # client retried the same request_id
print("Naive ledger (BUG — refunded twice):", refund_ledger_naive)


# --- the idempotent version: dedupe on the client's request id ---
refund_ledger_safe = []
seen_request_ids = set()

def idempotent_refund_handler(request_id: str, order_text: str):
    if request_id in seen_request_ids:
        return {"status": "issued", "note": "duplicate request ignored"}
    seen_request_ids.add(request_id)
    refund_ledger_safe.append(order_text)
    return {"status": "issued"}

idempotent_refund_handler("req-2", "order #901")
idempotent_refund_handler("req-2", "order #901")  # same retry
print("Safe ledger  (fixed — refunded once):", refund_ledger_safe)


Naive ledger (BUG — refunded twice): ['order #900', 'order #900']
Safe ledger  (fixed — refunded once): ['order #901']


## 10. Production Checklist

Everything above is spec-shaped but toy-sized. Before any of this touches
real money, real customers, or real infrastructure:

| # | Concern | What to actually do |
|---|---|---|
| 1 | **Idempotency** | Key on a client-supplied request id (§9 above), not just `task.id` |
| 2 | **Token scoping** | Issue tokens scoped to one skill, not "full agent access" |
| 3 | **Timeouts & cancellation** | Always honor `tasks/cancel`; don't let a task run forever |
| 4 | **Persistence** | Replace `InMemoryTaskStore` with Postgres/Redis — tasks must survive a restart |
| 5 | **Observability** | Log `task.id` + `context_id` on every hop for cross-agent tracing |
| 6 | **Schema validation per skill** | The spec doesn't standardize each skill's input shape — validate it yourself (Pydantic models per skill) |
| 7 | **Rate limiting** | A popular agent becomes a dependency for others — protect it like any public API |
| 8 | **Agent Card versioning** | Bump `version`; keep old skill ids alive during migrations |
| 9 | **TLS in transit** | Plain HTTP (as used in this notebook) is fine for learning, never for production |
| 10 | **Human-in-the-loop UX** | Decide what your product does when a task sits in `input-required` for hours |

None of these are exotic — they're the same discipline you'd apply to any
public HTTP API. A2A doesn't remove that responsibility; it just gives the
protocol shape everyone agrees to speak.


## Recap

```
Agent Card  →  discovery ("here's what I can do")
Task        →  the unit of delegated work, with a state machine
Message     →  one turn of dialogue, made of Parts
Part        →  text / data / file — the atoms of content
Artifact    →  what the task actually produces
```

Everything in this notebook — the FastAPI app, the `httpx` client, the
orchestration `asyncio.gather` — is a fully general pattern. Point the
`base_url` at a real deployed agent instead of an in-process ASGI transport,
and the exact same client code talks to it over the real network.
